# UK Exports Predictor

This notebook is the project write-up. It learns from global HS6 goods trade what Flow to expect between any Exporter, Importer and Product, applies that to the UK to estimate **Export Potential** for every Product–Market pair, and ranks positive **Export Gaps** as **Opportunities**.

All reusable logic lives in the `exportspredictor` package, so the notebook and the API compute features identically. The notebook only calls it.

## Parameters

`FIXTURE_MODE = True` runs everything on a tiny synthetic world in seconds; the test suite executes the notebook this way. With `FIXTURE_MODE = False`, section 1 downloads BACI (once) into the data folder set by `EXPORTSPREDICTOR_DATA_DIR` and explores the real Flows; real-data training arrives with the feature-table work.

In [ ]:
FIXTURE_MODE = True

In [ ]:
import polars as pl

from exportspredictor import artefacts, baci, policy
from exportspredictor.fixture import build_world
from exportspredictor.paths import artefacts_dir
from exportspredictor.pipeline import build_artefacts

## 1. The Flows: BACI HS12

The training data is CEPII's BACI, HS 2012 revision, release V202601 (ADR 0003): every non-zero Flow between about 226 Exporters and Importers for about 5,200 Products, 2012–2024. BACI reconciles each country's export report with its partners' import reports, so a UK Flow is a blend of what the UK and the Market declared, not HMRC's figure. Values are converted from thousand USD to USD.

This section is the course's **exploratory data analysis** and **data-quality audit**, applied at a scale where pandas would not fit: the download is converted once from CSV to Parquet with compact types (Enum codes for countries and Products, so `010121` keeps its leading zero), and Polars scans it lazily, so only the columns and years a query needs are read. The conversion peaks at about 2.5 GB of memory.

In fixture mode the same summaries run on the fixture world's Flows.

In [ ]:
if FIXTURE_MODE:
    flows = build_world().flows.lazy()
else:
    print(baci.prepare())  # idempotent: skips the download and conversion once done
    flows = baci.scan_flows()
flows.head(5).collect()

### Coverage and size per year

Row counts per year, with the number of Products, Exporters and Importers that trade, total world value and the UK's share of it. BACI records only strictly positive Flows, so a missing row means zero *or* unreported: the model treats absent pairs as zeros, which matters for poorly reporting countries.

In [ ]:
flow_summary = baci.summarise_flows(flows)
flow_summary

In [ ]:
flows.filter(pl.col("value_usd") > 0).select(
    flows=pl.len(),
    products=pl.col("hs6").n_unique(),
    exporters=pl.col("exporter").n_unique(),
    importers=pl.col("importer").n_unique(),
).collect()

On the real data (V202601) there are about 9.0 M Flows in 2012, rising to 11.8 M in 2023, about 142 M in total. 5,199 of the 5,202 HS 2012 Products trade in a typical year, and 226 Exporters appear each year (227 across all years). The UK's share of world goods exports in BACI drifts down from about 2.4–2.6% in 2012–2016 to about 1.9% in 2021–2023.

Flow values are extremely skewed: the mean UK Flow (about $1.3 M) is roughly 100 times its median (about $14k), which is why the model uses a Tweedie loss rather than squared error on raw values (course: skewed variables favour the median).

In [ ]:
uk_latest = flows.filter((pl.col("exporter").cast(pl.String) == baci.UK) & (pl.col("value_usd") > 0))
uk_latest.group_by("year").agg(
    mean_usd=pl.col("value_usd").mean(),
    median_usd=pl.col("value_usd").median(),
    markets=pl.col("importer").n_unique(),
    products=pl.col("hs6").n_unique(),
).sort("year").collect()

### The provisional 2024 data

BACI's latest year is provisional: Comtrade is downloaded in January, before every country has reported, and CEPII revises it in the next release. The table below compares each year with the one before. On the real data, 2024 has about 5.5% fewer Flows than 2023 while world value is almost flat, consistent with late reporters still missing small Flows. UK exports fall about 8% in 2024, which may partly reflect the same incompleteness. 2024 is the Baseline year, so the app labels it provisional, and the evaluation (test year 2023) never relies on it.

In [ ]:
flow_summary.select(
    "year",
    flows_change=pl.col("flows").pct_change(),
    world_value_change=pl.col("world_value_usd").pct_change(),
    uk_value_change=pl.col("uk_value_usd").pct_change(),
).tail(3)

## 2. Policy features: Trade Agreements and Trading Blocs

The policy features are the only columns a Scenario changes, so they must be right for every year. They come from the WTO's Regional Trade Agreements database (RTA-IS), "Export all RTAs" file. It lists every notified agreement with its goods entry-into-force date and **per-member entry and exit dates** (`docs/research/trade-agreement-coverage.md`). CEPII Gravity, the usual source in trade economics, stops in 2021, so it misses UK–Australia, UK–New Zealand and the UK's CPTPP accession.

For each pair and year, we build:

- `trade_agreement`: a goods Trade Agreement is in force between the two countries;
- `customs_union`: the agreement is a Customs Union (WTO type "CU");
- `eu_o/_d`, `cptpp_o/_d`, `efta_o/_d`: Trading Bloc membership of the Exporter and the Importer.

Each rule below follows how the WTO and Gravity code agreements:

- **The 1 July rule** (Gravity's): a membership counts in a year if it began before 1 July and had not ended by then. EU–Japan (1 Feb 2019) counts from 2019, and UK–Australia and UK–NZ (31 May 2023) from 2023. The UK's CPTPP accession (15 Dec 2024) counts only from 2025, so the 2024 Baseline does not include it.
- **A bloc that is a party is expanded to its members at each date.** "EU – United Kingdom" lists only "European Union" as the EU side. Croatia is still covered, because it is an EU member. An "EU – X" agreement links the EU side with X, not X's partners with each other.
- **The UK in 2020:** the UK left the EU (the Trading Bloc) on 31 Jan 2020. It stayed in the EU Customs Union and the EU's agreements until 31 Dec 2020 (the transition period). So for 2020 the UK's `eu_o` is 0, while its `customs_union` with EU members is still 1. The "EU Customs Union only" Scenario relies on this distinction.
- **Partial-scope agreements** (WTO type "PSA", such as the GSTP among developing countries) cut tariffs on short product lists only, so they do not count as Trade Agreements. Gravity's `fta_wto` leaves them out too.

This applies the course's **feature engineering** with explicit **data lineage**. Every flag traces back to a dated record in a primary source. `python -m exportspredictor.policy` rebuilds the tables from scratch; it is a step in `scripts/rebuild.py`.

In [ ]:
if FIXTURE_MODE:
    print("Fixture mode: the fixture world carries its own trade_agreement column (section 3).")
else:
    policy_files = policy.build()  # downloads the WTO export once, then rebuilds in seconds
    agreements = pl.read_parquet(policy_files["agreements"])
    trading_blocs = pl.read_parquet(policy_files["trading_blocs"])
    print(agreements.height, "pair-years with a Trade Agreement;", trading_blocs.height, "country-years in a tracked Trading Bloc")

The UK's policy features for a few Markets show the known dates:

- the EU Customs Union until 2020;
- the UK–EU agreement from 2021;
- Australia and New Zealand from 2023;
- no CPTPP membership yet in 2024.

In [ ]:
if not FIXTURE_MODE:
    uk_rows = pl.DataFrame(
        {"exporter": "GBR", "importer": ["FRA", "JPN", "AUS", "NZL", "MEX", "USA"]}
    ).join(pl.DataFrame({"year": list(range(2018, 2025))}, schema={"year": pl.Int32}), how="cross")
    uk_policy = policy.attach_policy(uk_rows, agreements, trading_blocs)
    display(uk_policy.pivot(on="year", index="importer", values="trade_agreement"))
    display(uk_policy.pivot(on="year", index="importer", values="customs_union"))
    display(uk_policy.filter(pl.col("importer") == "FRA").select("year", "eu_o", "cptpp_o", "efta_o"))

### Cross-check against CEPII Gravity, 2012–2021

Gravity V202211 codes `fta_wto` (per pair) and `eu_o/_d` (per country) up to 2021. It uses the same WTO database, with corrections by Thierry Mayer. We compare every pair-year that Gravity has. Most pairs have no agreement in either source, so the plain agreement rate flatters us. We therefore also report `positive_overlap`: of the pair-years that either source codes as in an agreement, the share that both do (the Jaccard index).

In [ ]:
if not FIXTURE_MODE:
    gravity_policy = policy.read_gravity_policy()  # reads the release zip once, then a cached extract
    crosscheck, disagreements = policy.gravity_crosscheck(agreements, trading_blocs, gravity_policy)
    display(crosscheck)

These are the results on the WTO export downloaded on 16 Nov 2026:

- **Trade Agreement:** the agreement rate is 95.5% over 549,900 pair-years, and the positive overlap is 70.6%. Both sources code 58,938 pair-years; 21,426 are ours only and 3,162 are Gravity only.
- **EU membership:** the agreement rate is 99.96%. The one difference is Croatia in 2013. Croatia joined on 1 July 2013, which is not *before* 1 July, so here it counts from 2014. Gravity counts any accession during the year.

Most of the **ours only** pair-years have a clear source:

- **Countries that Gravity never codes in any agreement (14%).** Gravity has no agreement at all for Montenegro, Bermuda or Yemen, and almost none for Liechtenstein. Yet the WTO lists, for example, EU–Montenegro (2008) and the Pan-Arab FTA.
- **Romania before 2018 (5%).** Gravity codes Romania's EU agreements only from 2018, although Romania joined the EU in 2007. This is a gap in Gravity.
- **The African Continental FTA (22%).** The WTO records it in force from 30 May 2019. Gravity does not code it, perhaps because trading under it began only in 2021.
- **The EU's Economic Partnership Agreements with African and Caribbean states (20%):** Eastern and Southern Africa, SADC, CARIFORUM and Ghana. They are notified to the WTO and applied, but Gravity's corrected `fta_wto` leaves most of these pairs out.
- **The rest (39%):** mostly COMESA accessions, SADC, EU–Serbia and EU–Central America. Here Gravity's corrections drop members that the export lists without dates.

Most of the **Gravity only** pair-years also have a clear source:

- **Territories missing from the WTO member lists (17%).** Mayotte, Réunion and other French overseas regions are part of the EU, but the WTO lists only France.
- **Haiti (17%).** Haiti signed the CARIFORUM agreement but has not ratified it. The WTO gives its entry date as 9999.
- **Latin American agreements notified as partial scope under LAIA**, such as Argentina–Bolivia and Argentina–Chile. We exclude these because they are PSAs.

The changes that the Scenarios and the Event Backtest rely on agree in both sources: EU–Japan from 2019, the UK's EU Customs Union to 2020, UK–EU and UK–Japan in 2021. Japan's only disagreements are CPTPP with Canada and New Zealand (in force 30 Dec 2018) and Japan–Mongolia (2016), which Gravity does not code.

UK pairs account for 368 of the disagreeing pair-years: 252 are ours only and 116 Gravity only. They are of the kinds above: Romania, Montenegro and Liechtenstein, the EPA partners, and a year's difference in when EU–Central America started. One more kind appears only for the UK. In 2021, Gravity keeps the UK in EU agreements that the UK did not roll over at the start of 2021, for example with Algeria, Syria, Andorra and the overseas territories. The WTO lists no UK agreement with these partners.

In [ ]:
if not FIXTURE_MODE:
    display(
        disagreements.filter(pl.col("feature") == "trade_agreement")
        .explode("years")
        .group_by("ours", "gravity")
        .agg(pair_years=pl.len(), countries=pl.concat_list("exporter", "importer").explode().n_unique())
    )
    display(disagreements.filter((pl.col("exporter") == "GBR") | (pl.col("importer") == "GBR")).head(20))

## 3. Tracer bullet: the fixture world

Before any real data, we prove that every layer connects: data, features, fold models, artefacts, API and app. The fixture world has 4 Exporters (including the UK), 3 Markets, 5 Products and 4 years. Flows follow a gravity-like rule, and the UK–Japan Trade Agreement switches on in 2022, so the world contains a policy change for the model to learn from.

In [ ]:
if not FIXTURE_MODE:
    raise NotImplementedError("Real-data mode is not available yet; set FIXTURE_MODE = True.")
world = build_world()
world.flows.head()

In [ ]:
world.policy.filter(pl.col("exporter") == "GBR").pivot(on="year", index="importer", values="trade_agreement")

## 4. Training and out-of-fold scoring

The model is gradient boosting (XGBoost) with a Tweedie objective, which suits a non-negative target with many zeros. This extends the course's tree-ensemble material. Three guardrails are already in place:

- **Time split** (course: train/validation/test splits and leakage): fold models fit on years before the Baseline year and score the Baseline year.
- **Out-of-fold UK scoring** (ADR 0001, beyond the course's k-fold cross-validation): each UK Product–Market pair gets its Export Potential from the fold model that never saw it, so the Export Gap is not shrunk by memorisation.
- **Monotone constraint** on the Trade Agreement feature (ADR 0002): adding a Trade Agreement never lowers Export Potential. The model *associates* an agreement with higher Flows; it does not show cause.

In [ ]:
fixture = build_artefacts(world, source="fixture")
fixture.metadata

The Baseline table compares Export Potential with Actual Exports for every UK pair; a positive Export Gap is an Opportunity.

In [ ]:
fixture.baseline.sort("export_gap_usd", descending=True)

## 5. Saving the artefacts

The API serves exactly what was evaluated here: a versioned artefact folder with the fold models, the Baseline table, lookups and a metadata file.

In [ ]:
saved_to = artefacts.save(fixture, artefacts_dir())
saved_to